# Surveillance des Sites Miniers et Activités Extractives par IA

## Introduction
La République Démocratique du Congo possède des zones minières vastes et parfois difficiles d'accès. Ce notebook démontre comment l'IA peut isoler les signatures anthropiques des sites miniers (sols décapés, bassins de rejets, routes d'accès) pour automatiser le suivi environnemental et réglementaire.

## Objectifs
*   **Séparation sol nu / forêt** : Identifier les zones de décapage minier.
*   **Suivi de l'emprise** : Calculer la superficie occupée par les activités minières.
*   **Détection proactive** : Repérer les nouvelles activités de creusage artisanal.

## Méthodologie
1.  **Setup** : Initialisation TerraTorch.
2.  **Acquisition** : Images Sentinel-2 (temporellement filtrées).
3.  **Encodage contextuel** : Utilisation de Prithvi pour différencier une route ou une carrière d'un sol nu naturel.
4.  **Classification** : Regroupement par K-Means pour isoler les clusters 'miniers'.

In [ ]:
# ====================================================
# ÉTAPE 1 : Setup
# ====================================================
!pip install geemap earthengine-api scikit-learn rasterio terratorch torch matplotlib seaborn -q

import ee, geemap, torch, rasterio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import MiniBatchKMeans
from terratorch import BACKBONE_REGISTRY

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print('✅ Système prêt')

## Zone d'Étude (ROI)
Focus sur une zone d'activité minière connue ou suspectée au Congo.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d'étude')
Map

## Acquisition des Données
Extraction des données Sentinel-2 les plus récentes.

In [ ]:
# ====================================================
# ÉTAPE 3 : Données Sentinel-2
# ====================================================
image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(roi).filterDate('2023-01-01', '2023-12-31')
         .median().clip(roi))

geemap.ee_export_image(image.select(['B2','B3','B4','B8','B11','B12']), 'mines.tif', scale=30, region=roi)

## Inférence et Identification des Clusters Miniers
L'IA analyse le paysage. Le clustering regroupe les pixels ayant des signatures de 'sol décapé industriel' ou d''infra-structure minière'.

In [ ]:
# ====================================================
# ÉTAPE 4 : Inférence et Clustering
# ====================================================
model = BACKBONE_REGISTRY.build('prithvi_eo_v2_300', num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')
with rasterio.open('mines.tif') as src: img = src.read().astype(np.float32) / 10000.0

with torch.no_grad():
    out = model(torch.from_numpy(img).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out

feats_np = feats[0, 1:].numpy()
kmeans = MiniBatchKMeans(n_clusters=6, n_init=3).fit(feats_np)
mining_labels = kmeans.labels_.reshape(int(np.sqrt(len(feats_np))), -1)

plt.figure(figsize=(10, 8))
plt.imshow(mining_labels, cmap='tab10')
plt.colorbar(label='Clusters / Unités de terrain')
plt.title('Segmentation Opérationnelle des Sites Miniers')
plt.axis('off')
plt.show()